# 1) Data Exploration

In [ ]:
import urllib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import PIL

from google.cloud import bigquery

from sklearn import set_config
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, MinMaxScaler
from sklearn.linear_model import LinearRegression, Ridge

set_config(display='diagram')

## 1.1) Load Data

**All data**

The data comes from this public Big Query dataset `wagon-public-datasets.taxifare`. There are tables with different dataset sizes. 

**Training dates**

- We train the model on data up to `2015-01-01`. We can't access any data beyond that date to train our model.
- For this first iteration, we train the ML model on a **200k randomly sampled subset** (so that everything fits in RAM memory on our laptops).
- We'll train on the full data once we have obtained funds to access some distributed computing.
- This data is available in the Big Query table `wagon-public-datasets.taxifare.raw_200k`.

Note to Engineering: The data is accessible publicly, but you need your own project to query it. Querying incurs a cost, that's why. Fortunately the dataset is small and querying it falls within the free tier that Google offers for BigQuery.

**Fill your `GCP_PROJECT` below 👇**, Then query the historical data (pre-2015), ordered by date (so as to train/test split chronologically more easily) !

To obtain your project id: run `gcloud config get project` in the terminal.

In [ ]:
GCP_PROJECT = "<your gcp project id>"

In [ ]:
GCP_PROJECT_WAGON = "wagon-public-datasets"
BQ_DATASET = "taxifare"
RAW_TABLE = "raw_200k"

MIN_DATE = '2009-01-01'
MAX_DATE = '2015-01-01'

COLUMN_NAMES_RAW = (
    'fare_amount',	
    'pickup_datetime', 
    'pickup_longitude', 
    'pickup_latitude', 
    'dropoff_longitude', 
    'dropoff_latitude', 
    'passenger_count'
)


In [ ]:
query = f"""
    SELECT {", ".join(COLUMN_NAMES_RAW)}
    FROM `{GCP_PROJECT_WAGON}`.{BQ_DATASET}.{RAW_TABLE}
    WHERE pickup_datetime BETWEEN '{MIN_DATE}' AND '{MAX_DATE}'
    ORDER BY pickup_datetime
    """
print(query)


In [ ]:
client = bigquery.Client(project=GCP_PROJECT)
query_job = client.query(query)
result = query_job.result()
df = result.to_dataframe()
df

In [ ]:
df.info()


Data types look all good, as expected because we query from a BigQuery table, and the table schema had the correct data types.

## 1.2) Clean Data

In [ ]:
df.shape


In [ ]:
df.describe()


Reflections:
1. There seem to be negative fare amounts?
1. Maximum fare amount 208 USD?
1. Passenger count max seems fine. But the minimum: zero passengers?
1. Minimum and maximum longitude and latitudes make no sense.

Note: less than 200k because we only use data pre-2015 for this initial run.

Remove duplicates

In [ ]:
df = df.drop_duplicates()
df.shape


Remove buggy transactions

In [ ]:
df = df.dropna(how='any', axis=0)
df.shape

Address issues 1, 2, and 3: negative and too high fares and no passengers:

In [ ]:
df = df[df.fare_amount > 0]
df = df[df.fare_amount < 100]
df = df[df.passenger_count > 0]
df.shape


Remove geographically irrelevant transactions (rows)

In [ ]:
# Let's check NYC bounding boxes
# Load image of NYC map
url = 'https://wagon-public-datasets.s3.amazonaws.com/data-science-images/07-ML-OPS/nyc_-74.3_-73.7_40.5_40.9.png'
nyc_map = np.array(PIL.Image.open(urllib.request.urlopen(url)))

# Bounding boxes coordinates of the image
bounding_box = (-74.3, -73.7, 40.5, 40.9)

plt.imshow(
    nyc_map,
    extent=bounding_box,
);


In [ ]:
df = df[df["pickup_latitude"].between(left=40.5, right=40.9)]
df = df[df["dropoff_latitude"].between(left=40.5, right=40.9)]
df = df[df["pickup_longitude"].between(left=-74.3, right=-73.7)]
df = df[df["dropoff_longitude"].between(left=-74.3, right=-73.7)]


In [ ]:
df.describe()


Seems all good now. Good enough to get started.

## 1.3) Visualize Data

In [ ]:
# Plot histogram of fare
df.fare_amount.hist(bins=100, figsize=(14,3))
plt.xlabel('fare $USD')
plt.title('Distribution of fares')


### 1.4) Baseline Score

A baseline model with the most obvious feature: the distance between `pickup` and `dropoff`.

Using the "Manhattan distance" (L1 distance), which computes the sum of horizontal and vertical distances between two points, instead of the diagonal (Euclidean, L2) distance. This seems closer to the reality of New York's street layout.

In [ ]:
def manhattan_distance_vectorized(df: pd.DataFrame, start_lat: str, start_lon: str, end_lat: str, end_lon: str) -> dict:
    """
    Calculate the Manhattan distance in km between two points on the earth (specified in decimal degrees).
    Vectorized version for pandas df
    """
    earth_radius = 6371

    lat_1_rad, lon_1_rad = np.radians(df[start_lat]), np.radians(df[start_lon])
    lat_2_rad, lon_2_rad = np.radians(df[end_lat]), np.radians(df[end_lon])

    dlon_rad = lon_2_rad - lon_1_rad
    dlat_rad = lat_2_rad - lat_1_rad

    manhattan_rad = np.abs(dlon_rad) + np.abs(dlat_rad)
    manhattan_km = manhattan_rad * earth_radius

    return manhattan_km

In [ ]:
distances = manhattan_distance_vectorized(
    df, 
    "pickup_latitude", 
    "pickup_longitude",
    "dropoff_latitude", 
    "dropoff_longitude"
)
distances

In [ ]:
distances.hist(bins=50)
plt.xlabel('distance (km)')
plt.title("Distribution of taxi ride distance");

# 2) Train/Val/Test Split

🚨 We're dealing with timestamped data:
- We need to split train/val/test in a **chronological** manner.
- We don't want to hold too long val or tests sets: macro-economic conditions are changing fast in real life!
- The smaller the `split_ratio`, the more often we'll have to re-train our model
- With a dataset of 6 years, it's perfectly fine to keep 1 month ahead of val, 1 month ahead for test
- In production, this means we shouldn't trust our model performance to extend beyond 1 month in the future

In [ ]:
split_ratio = 0.02 # ~1 month for val, ~1 month for test

test_length = int(len(df) * split_ratio)
val_length = int((len(df)-test_length) * split_ratio)
train_length = len(df) - val_length - test_length

df_train = df.iloc[:train_length, :].sample(frac=1) # Shuffle datasets to improve training
df_val = df.iloc[train_length: train_length + val_length, :].sample(frac=1)
df_test = df.iloc[train_length+val_length:, :].sample(frac=1)

print(df_train.shape)
print(df_val.shape)
print(df_test.shape)

assert len(df_train) + len(df_val) + len(df_test) == len(df)

In [ ]:
print("Train set date range:")
print(df_train.pickup_datetime.min())
print(df_train.pickup_datetime.max())
print('---')
print("Val set date range:")
print(df_val.pickup_datetime.min())
print(df_val.pickup_datetime.max())
print('---')
print("Test set date range:")
print(df_test.pickup_datetime.min())
print(df_test.pickup_datetime.max())

In [ ]:
X = df.drop("fare_amount", axis=1)
y = df[["fare_amount"]]

X_train = df_train.drop("fare_amount", axis=1)
y_train = df_train["fare_amount"]

X_val = df_val.drop("fare_amount", axis=1)
y_val = df_val["fare_amount"]

X_test = df_test.drop("fare_amount", axis=1)
y_test = df_test["fare_amount"]

### Simple Baseline: Linear Regression based on distance alone

In [ ]:
distances_train = manhattan_distance_vectorized(X_train, "pickup_latitude", "pickup_longitude","dropoff_latitude", "dropoff_longitude")
distances_val = manhattan_distance_vectorized(X_val, "pickup_latitude", "pickup_longitude","dropoff_latitude", "dropoff_longitude")
distances_test = manhattan_distance_vectorized(X_test, "pickup_latitude", "pickup_longitude","dropoff_latitude", "dropoff_longitude")

# Convert Series to DataFrame for easier handling
distances_train = pd.DataFrame(distances_train)
distances_val = pd.DataFrame(distances_val)
distances_test = pd.DataFrame(distances_test)


In [ ]:
sns.regplot(x=distances_train, y=y_train)
plt.xlabel("distance");


In [ ]:
baseline_model = LinearRegression()
baseline_model.fit(pd.DataFrame(distances_train), y_train)


In [ ]:
baseline_pred_val = baseline_model.predict(distances_val)
baseline_pred_test = baseline_model.predict(distances_test)

baseline_mae_val = np.mean(np.abs(baseline_pred_val - y_val))
baseline_mae_test = np.mean(np.abs(baseline_pred_test - y_test))

print(f'mean taxifare prices on train set = {round(np.mean(y_train), 2)} $')

print(f'🎯 baseline MAE on val set = {round(baseline_mae_val, 2)} $')
print(f'🎯 baseline MAE on test set = {round(baseline_mae_test, 2)} $')


# 3) Preprocessing Pipeline

In [ ]:
set_config(transform_output="pandas")

Note to self: Setting scikit-learn to always output transformed data as Pandas DataFrames. With the small sample this should work fine. The preprocessor is one-hot encoding several columns with many columns, so we'll have lots of `0`s in the outputs. When moving to the larger dataset, I'll have to disable this.

Note to Engineering: This might impact the saved pipeline, so you might have to adapt whatever you are developing afterwards. Let me know.

The dataset has only 5 features (passengers + lon/lat), and at least hundreds of thousands of lines, even with the small sample.

👉 In the next part we "engineer" extra features such as "hour of the day"  
- Adding them will not cause any problem because the huge number of rows will allow our model to learn all weights associated with these multiple features.

❗️ The proposed preprocessor below outputs a **fixed number of features** that is **independent of the training set**. We avoid potential statistical differences due to randomness in the train-test-split by normalizing using fixed numbers, rather than using scalers that depend on statistical measures.

## 3.1) Passenger Preprocessors

Let's analyze passenger numbers

In [ ]:
sns.histplot(data=df, x="passenger_count");


In [ ]:
# Passenger count pipe
def scale_passenger(p):
    p_min = 0.
    p_max = 8.
    p_scaled = (p - p_min) / (p_max - p_min)
    return p_scaled

passenger_pipe = FunctionTransformer(scale_passenger)

In [ ]:
passenger_pipe.fit_transform(X_train[['passenger_count']]).value_counts()


## 3.2) Time Preprocessor

Let's extract some interesting attributes from the `pickup_datetime`
- hour of the day
- day of the week
- month of the year
- number of days since 2009 (this is way to encode inflation influence)

In [ ]:
def transform_time_features(X: pd.DataFrame) -> pd.DataFrame:
    timedelta = (X["pickup_datetime"] - pd.Timestamp('2009-01-01T00:00:00', tz='UTC')) / pd.Timedelta(1,'D')

    pickup_dt = X["pickup_datetime"].dt.tz_convert("America/New_York").dt

    dow = pickup_dt.weekday
    hour = pickup_dt.hour
    month = pickup_dt.month

    hour_sin = np.sin(2 * math.pi / 24 * hour)
    hour_cos = np.cos(2 * math.pi / 24 * hour)

    month_sin = np.sin(2 * math.pi / 24 * month)
    month_cos = np.cos(2 * math.pi / 24 * month)

    return pd.DataFrame({
        'hour_sin': hour_sin,
        'hour_cos': hour_cos,
        'day_of_week': dow,
        'month_sin': month_sin,
        'month_cos': month_cos,
        'timedelta': timedelta
    })

X_time_processed = transform_time_features(X[["pickup_datetime"]])
X_time_processed

Hour and month are temporal features. Cyclical features like time need some specific preprocessing.

<img src="https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/02-Prepare-the-dataset/cyclical_feature_engineering.png" alt="Cyclical features" width="1000">

For reference: this [article](https://ianlondon.github.io/posts/encoding-cyclical-features-24-hour-time/) for more details.

Now one-hot-encode `"day of week"` by forcing all 7 categories to be always present in `X_processed` , independent of the sampling. We can do this because we now upfront which days of the week exist. No need to derive it from the data.

This will give us a fixed size of columns for `X_processed` at the end.

In [ ]:
days_of_week = np.arange(0, 7, 1)  # days of the week from 0 to 6

day_of_week_encoder = OneHotEncoder(
    categories=[days_of_week], 
    handle_unknown="ignore",
    sparse_output=False
)

day_of_week_encoder.fit_transform(X_time_processed[['day_of_week']])


And combine this with a sort of "Min-Max" re-scaling of the `timedelta` column

In [ ]:
print(X_time_processed['timedelta'].min())
print(X_time_processed['timedelta'].max())


In [ ]:
timedelta_min = 0
timedelta_max = 2190 # Our model may extend in the future. No big deal if the scaled data extend slightly beyond 1.0


In [ ]:
def scale_timedelta(timedelta):
    scaled = (timedelta - timedelta_min) / (timedelta_max - timedelta_min)
    return scaled

Time pipeline

In [ ]:
time_pipe = make_pipeline(
    FunctionTransformer(transform_time_features),
    make_column_transformer(
        (day_of_week_encoder, ["day_of_week"]),
        (FunctionTransformer(scale_timedelta), ["timedelta"]),
        remainder="passthrough" # keep hour_sin and hour_cos, and month_sin and month_cos
    )
)


☝️ All features approximately centered and scaled

## 3.3) Distance Pipeline

Let's add both the Manhattan distances as feature

In [ ]:
lonlat_features = ["pickup_latitude", "pickup_longitude", "dropoff_latitude", "dropoff_longitude"]

In [ ]:
def manhattan_distance_for_pipe(df):
    distance = manhattan_distance_vectorized(df, *lonlat_features)
    return pd.DataFrame({'distance': distance})

In [ ]:
def scale_distance(dist):
    dist_min = 0
    dist_max = 100
    scaled = (dist - dist_min) / (dist_max - dist_min)
    return scaled

In [ ]:
distance_pipe = make_pipeline(
    FunctionTransformer(manhattan_distance_for_pipe),
    FunctionTransformer(scale_distance)
    )
distance_pipe

distance_pipe.fit_transform(X_train[lonlat_features])

## 3.5) Full Preprocessing Pipeline

Complete preprocessor

In [ ]:
preprocessor = ColumnTransformer(
    [
        ("passenger_preproc", passenger_pipe, ["passenger_count"]),
        ("time_preproc", time_pipe, ["pickup_datetime"]),
        ("dist_preproc", distance_pipe, lonlat_features),
    ],
)

preprocessor

In [ ]:
preprocessor.fit_transform(X_train).describe()

☝️ The preprocessor outputs a **fixed** number of features that is independent of the training set. 

☝️ The preprocessor is also  **state-less** (i.e it has no `.fit()` method, only a `.transform()`). It can be seen as a *pure function* $f:X \rightarrow X_{processed}$ without an internal state, as opposed to standard scaling for instance, which has to store "X_train standard deviations" as internal states.

These two features will make work much easier for the ML Engineering team to scale preprocessing to hundreds of GBs. 

# 4) Model

## 4.1) Architecture

In [ ]:
linreg = Ridge()

linreg_pipeline = make_pipeline(
    preprocessor, linreg
)

In [ ]:
linreg_pipeline.fit(X_train, y_train)

## 4.2) Performance evaluation

In [ ]:
linreg_pred_val = linreg_pipeline.predict(X_val)
linreg_pred_test = linreg_pipeline.predict(X_test)

linreg_mae_val = np.mean(np.abs(linreg_pred_val - y_val))
linreg_mae_test = np.mean(np.abs(linreg_pred_test - y_test))

print(f'mean taxifare prices on train set = {round(np.mean(y_train), 2)} $')

print(f'🎯 linreg MAE on val set = {round(linreg_mae_val, 2)} $')
print(f'🎯 linreg MAE on test set = {round(linreg_mae_test, 2)} $')

Note to self: Still need to evaluate residuals. But maybe first try to use a better model. Seems the linear regression is not doing great, not even with all the features.